# Regresión lineal múltiple usando matrices

*¿Cómo llevamos las fórmulas de MCO a R?*

Calcularemos los coeficientes, sus errores estándar y la bondad de ajuste del modelo de ventas, precio y publicidad que estudiamos en clase.

## Importamos los datos

Usamos la base de Big Andy's Burger Barn de Hill, Griffiths y Lim (2011): 75 observaciones de franquicias en distintas ciudades. `sales` representa los ingresos mensuales por ventas y `advert` el gasto mensual en publicidad, ambos en miles de dólares. `price` es un índice de precios de los productos, expresado en dólares.

Usamos read_csv() del paquete readr para cargar la base desde una ruta relativa. La ruta es relativa a la carpeta del notebook. Mostramos las primeras observaciones.

In [1]:
library(readr)

datos <- read_csv("../data/andy.csv")
head(datos)

Rows: 75 Columns: 3


── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (3): sales, price, advert



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


sales,price,advert
<dbl>,<dbl>,<dbl>
73.2,5.69,1.3
71.8,6.49,2.9
62.4,5.63,0.8
67.4,6.22,0.7
89.3,5.02,1.5
70.3,6.41,1.3


## Planteamos el modelo

Queremos estimar

$$
sales_i=\beta_1+\beta_2price_i+\beta_3advert_i+u_i.
$$

Cada pendiente representa una relación parcial: el cambio en las ventas esperadas asociado con un cambio en un regresor, manteniendo constante el otro.

## Construimos las matrices

Escribimos el modelo como $\mathbf y=\mathbf X\boldsymbol\beta+\mathbf u$. La primera columna de $\mathbf X$ contiene unos y corresponde a la constante; las otras dos contienen precio y publicidad.

`matrix()` construye una matriz y `cbind()` une columnas. `n` es el número de observaciones y `k` el número de coeficientes, incluida la constante.

In [2]:
n <- nrow(datos)
y <- matrix(datos$sales)
o <- matrix(rep(1, n))
x <- cbind(o, datos$price, datos$advert)
colnames(x) <- c("(Intercept)", "price", "advert")
k <- ncol(x)

head(x)
c(n = n, k = k)

(Intercept),price,advert
1,5.69,1.3
1,6.49,2.9
1,5.63,0.8
1,6.22,0.7
1,5.02,1.5
1,6.41,1.3


n  k 
75  3

## Estimamos los coeficientes

$$
\mathbf b=(\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y.
$$

`t()` transpone una matriz, `%*%` multiplica matrices y `solve()` obtiene la inversa. En esta base, las columnas de $\mathbf X$ son linealmente independientes, por lo que $\mathbf X^\top\mathbf X$ es invertible.

In [3]:
xx <- t(x) %*% x
xy <- t(x) %*% y
b <- solve(xx) %*% xy
b

(Intercept),118.913610
price,-7.907854
advert,1.862584


## Valores ajustados y residuos

$$
\hat{\mathbf y}=\mathbf X\mathbf b,
\qquad
\mathbf e=\mathbf y-\hat{\mathbf y}.
$$

Los residuos son diferencias entre los valores observados y los ajustados; los errores $u_i$ del modelo no son observables.

In [4]:
yhat <- x %*% b
e <- y - yhat
head(cbind(Observado = datos$sales, Ajustado = as.numeric(yhat),
           Residuo = as.numeric(e)))

Observado,Ajustado,Residuo
73.2,76.33928,-3.1392788
71.8,72.99313,-1.1931302
62.4,75.88246,-13.4824580
67.4,71.03057,-3.6305655
89.3,82.01006,7.2899419
70.3,70.64562,-0.3456237


## Suma de cuadrados de los residuos

$$
RSS=\mathbf e^\top\mathbf e.
$$

El producto devuelve una matriz de $1\times1$. Usamos `as.numeric()` para guardar su único elemento como un número.

In [5]:
RSS <- as.numeric(t(e) %*% e)
RSS

[1] 1718.943

## Estimamos la varianza de los errores

$$
s^2=\frac{RSS}{n-k}.
$$

Los grados de libertad son $n-k$: hemos estimado $k$ coeficientes, incluida la constante.

In [6]:
gl <- n - k
s2 <- RSS / gl
c(gl = gl, s2 = s2)

gl       s2 
72.00000 23.87421

## Matriz de varianzas y covarianzas de los estimadores

Bajo los supuestos de homoscedasticidad y ausencia de correlación entre errores,

$$
\widehat{\operatorname{Var}}(\mathbf b)
=s^2(\mathbf X^\top\mathbf X)^{-1}.
$$

La diagonal contiene las varianzas estimadas de los coeficientes y los elementos fuera de ella contienen sus covarianzas. `*` multiplica cada elemento de la matriz por el escalar `s2`.

In [7]:
Vb <- s2 * solve(xx)
Vb

,(Intercept),price,advert
(Intercept),40.3432990,-6.79506412,-0.74842060
price,-6.7950641,1.20120070,-0.01974215
advert,-0.7484206,-0.01974215,0.46675606


## Errores estándar

Los errores estándar son las raíces cuadradas de las varianzas estimadas. `diag()` extrae la diagonal principal de `Vb`.

In [8]:
se <- sqrt(diag(Vb))
se

(Intercept)       price      advert 
  6.3516375   1.0959930   0.6831955

## Suma de cuadrados total y $R^2$

$$
TSS=(\mathbf y-\bar y\mathbf 1)^\top(\mathbf y-\bar y\mathbf 1),
\qquad
R^2=1-\frac{RSS}{TSS}.
$$

$R^2$ mide la proporción de la variación muestral de las ventas alrededor de su media que explica el modelo con constante.

In [9]:
ys <- y - mean(y)
TSS <- as.numeric(t(ys) %*% ys)
R2 <- 1 - RSS / TSS
c(TSS = TSS, R2 = R2)

TSS           R2 
3115.4818667    0.4482578

## $R^2$ ajustada

$$
\bar R^2=1-(1-R^2)\frac{n-1}{n-k}.
$$

La medida ajustada toma en cuenta el número de coeficientes estimados y puede disminuir al agregar regresores.

In [10]:
RA <- 1 - (1 - R2) * (n - 1) / (n - k)
c(R2 = R2, R2_ajustada = RA)

R2 R2_ajustada 
  0.4482578   0.4329316

## Interpretación

El coeficiente de `price` es aproximadamente −7.908: para un mismo gasto en publicidad, un aumento de un dólar en el índice de precios se asocia con una disminución de aproximadamente 7.908 miles de dólares en las ventas esperadas. El coeficiente de `advert` es aproximadamente 1.863: para un mismo precio, mil dólares adicionales de publicidad se asocian con un aumento de aproximadamente 1.863 miles de dólares en las ventas esperadas.

El modelo explica cerca del 44.8% de la variación muestral de las ventas. Estos cálculos describen relaciones estimadas; por sí solos no establecen efectos causales.

En el siguiente notebook obtendremos estos resultados con `lm()`.